# TOPIC 10 – ĐO LƯỜNG RỦI RO DANH MỤC VN30 BẰNG DCC-GARCH
## Thành viên 2 – Như: GARCH biên (univariate)
Chạy **sau** TV1. Chạy bằng **Restart Kernel → Run All**. Thứ tự chung của cả nhóm: TV1 → TV2 → TV3 → TV4 → TV5.

| | |
|---|---|
| Đọc từ | `TV1_Data_EDA/log_returns.csv` |
| Ghi ra | `TV2_GARCH/` (`std_resid.csv`, `cond_vol.csv`, `garch_selected_models.csv`, `TV2_garch_results.xlsx`, hình) |
| Cấu hình | ô 0.2 (đường dẫn, hằng số) – **giống hệt ở cả 5 notebook**, chỉ cần sửa `PROJECT_PATH` / `OUTPUT_PATH` cho đúng máy |

Thư viện: numpy, pandas, scipy, matplotlib, openpyxl (statsmodels tuỳ chọn). Xem `README.md`.

In [ ]:
# ===================== 0.1 THƯ VIỆN =====================
import os                     # đọc biến môi trường
import re                     # biểu thức chính quy: tách dòng cổ tức / chia tách trong file Yahoo
import warnings               # tắt cảnh báo không cần thiết
import datetime as dt         # kiểu ngày tháng
from itertools import combinations   # tạo các cặp cổ phiếu
from pathlib import Path      # thao tác đường dẫn an toàn trên Windows

import numpy as np            # tính toán ma trận
import pandas as pd           # bảng dữ liệu
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats                     # phân phối Normal / t / chi-bình phương, JB
from scipy.optimize import minimize         # tối đa hoá hàm hợp lý (MLE)
from scipy.signal import lfilter            # lọc tuyến tính nhanh cho phương trình GARCH
from scipy.special import gammaln           # log Gamma cho mật độ Student-t

try:                                        # statsmodels: chỉ dùng cho ADF / KPSS (nếu có)
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.tools.sm_exceptions import InterpolationWarning
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:                                        # display() có sẵn trong Jupyter; chạy file .py thì thay bằng print
    display
except NameError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("statsmodels có sẵn:", HAS_SM)

In [ ]:
# ===================== 0.2 CẤU HÌNH DÙNG CHUNG CHO CẢ 5 THÀNH VIÊN =====================
# >>> ĐƯỜNG DẪN – sửa 2 dòng này nếu đổi máy / đổi thư mục <<<
PROJECT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo"          # thư mục dự án (chứa input/, ouput/, code/)
OUTPUT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo\ouput"    # nơi lưu toàn bộ kết quả


def find_base_dir() -> Path:
    """Tìm thư mục dự án: biến môi trường PROJECT_DIR -> PROJECT_PATH -> thư mục cha của code/."""
    env = os.environ.get("PROJECT_DIR")
    if env:
        return Path(env)
    if Path(PROJECT_PATH).exists():
        return Path(PROJECT_PATH)
    cwd = Path.cwd()
    return cwd.parent if cwd.name.lower() == "code" else cwd


def pick_dir(base: Path, names) -> Path:
    """Chọn thư mục con đã tồn tại (chấp nhận Input/input)."""
    return next((base / n for n in names if (base / n).exists()), base / names[0])


BASE_DIR = find_base_dir()
INPUT_DIR = pick_dir(BASE_DIR, ["input", "Input"])
if not list(INPUT_DIR.glob("*.xlsx")):                  # dự phòng: thư mục Input cũ của Project midterm
    old_in = Path(r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm\Input")
    if list(old_in.glob("*.xlsx")):
        INPUT_DIR = old_in
# Output: biến môi trường OUTPUT_DIR -> OUTPUT_PATH (nếu ổ D: tồn tại) -> BASE_DIR/ouput
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR") or
                  (OUTPUT_PATH if Path(OUTPUT_PATH).anchor and Path(Path(OUTPUT_PATH).anchor).exists()
                   else BASE_DIR / "ouput"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_OUT = OUTPUT_DIR                                    # mỗi thành viên một thư mục con trong ouput/
OUT = {k: ROOT_OUT / v for k, v in {"TV1": "TV1_Data_EDA", "TV2": "TV2_GARCH", "TV3": "TV3_DCC",
                                     "TV4": "TV4_VaR_Backtest", "TV5": "TV5_Euler_Stress"}.items()}
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Hằng số dùng chung ----
TICKERS_ORDER = ["FPT", "HPG", "MWG", "VCB", "VHM"]    # thứ tự hiển thị (mã khác trong Input vẫn được đọc)
ALPHA = 0.05                 # mức ý nghĩa của mọi kiểm định
LB_LAGS = [5, 10, 20]        # số trễ Ljung-Box
TRADING_DAYS = 252           # số phiên / năm để năm hoá
SCALE = 100.0                # ước lượng GARCH trên lợi suất % (r × 100) cho ổn định số học
SELECT_BY = "BIC"            # tiêu chí chọn mô hình GARCH: "AIC" hoặc "BIC"
PAIRS = [("VCB", "VHM"), ("HPG", "FPT")]               # cặp tương quan động theo đề
VAR_LEVELS = [0.95, 0.99]    # mức tin cậy VaR
HS_WINDOW = 250              # cửa sổ Historical Simulation (≈ 1 năm)
BASEL_WINDOW = 250           # cửa sổ Basel Traffic Light (250 phiên gần nhất)
VIOL_MODEL = "Normal"        # >>> MÔ HÌNH ĐÁNH DẤU VI PHẠM TRÊN MỌI HÌNH & DIỄN GIẢI TV4: "Normal" hoặc "t" <<<
VIOL_OOS = {"Normal": "DCC-N", "t": "DCC-t"}[VIOL_MODEL]   # tên tương ứng trong phần ngoài mẫu
EULER_CL = 0.99              # mức tin cậy dùng cho phân bổ rủi ro Euler
STRESS = {                   # giai đoạn khủng hoảng theo đề (TV5)
    "Covid-19 (02–04/2020)": ("2020-02-01", "2020-04-30"),
    "Trái phiếu & lãi suất (04–11/2022)": ("2022-04-01", "2022-11-30"),
}


STRESS_COLORS = ["tab:red", "tab:orange"]   # mỗi giai đoạn một màu để phân biệt Covid-19 với Trái phiếu & lãi suất


def shade_stress(ax, alpha=0.12, label=True, xlim=None):
    """Tô màu từng giai đoạn khủng hoảng (mỗi giai đoạn một màu, chú thích theo tên giai đoạn).
    xlim=(đầu, cuối): bỏ qua giai đoạn nằm hoàn toàn ngoài khoảng này (tránh kéo giãn trục thời gian)."""
    for (name, (s, e)), c in zip(STRESS.items(), STRESS_COLORS):
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        if xlim is not None and (e < xlim[0] or s > xlim[1]):
            continue
        ax.axvspan(s, e, color=c, alpha=alpha, label=(name if label else None))


def grid_axes(n, ncol=2, h=3.2, w=13, **kw):
    """Tạo lưới ô vẽ cho n mã, ẩn các ô thừa."""
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w, h * nrow), squeeze=False, **kw)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("BASE_DIR  :", BASE_DIR)
print("INPUT_DIR :", INPUT_DIR, "| tồn tại:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


def period_masks(index):
    """Mặt nạ True/False cho từng giai đoạn: các giai đoạn STRESS + 'Bình thường' (ngoài mọi giai đoạn stress)."""
    m = {name: (index >= s) & (index <= e) for name, (s, e) in STRESS.items()}
    m = {"Bình thường": ~np.any(np.column_stack(list(m.values())), axis=1), **m}
    return m


## Nạp kết quả của các thành viên trước

In [ ]:
# ===================== NẠP KẾT QUẢ TV1 + HÀM DÙNG CHUNG =====================
def need(path, prev):
    """Báo lỗi rõ ràng nếu chưa chạy notebook của thành viên trước."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Chưa có {path} – hãy chạy notebook {prev} trước (xem README.md).")
    return Path(path)


log_ret = pd.read_csv(need(OUT["TV1"] / "log_returns.csv", "TV1"), index_col=0, parse_dates=True, float_precision="round_trip")
log_ret.index.name = "Date"
tickers = list(log_ret.columns); N = len(tickers); T = len(log_ret)
print(f"[TV1] log_ret: {T} phiên × {N} mã ({', '.join(tickers)}) | {log_ret.index.min().date()} -> {log_ret.index.max().date()}")

# Hàm kiểm định tự tương quan (định nghĩa ở TV1, dùng lại để chẩn đoán phần dư ở mục 2.4)
def acf(x, nlags):
    """Hệ số tự tương quan mẫu ρ_1..ρ_nlags."""
    x = np.asarray(x, float); x = x - x.mean()
    denom = x @ x
    return np.array([(x[k:] @ x[:-k]) / denom for k in range(1, nlags + 1)])


def ljung_box(x, lags):
    """Ljung-Box Q(m) và p-value cho từng m trong lags."""
    x = np.asarray(pd.Series(x).dropna(), float)
    n = len(x)
    rho = acf(x, max(lags))
    out = {}
    for m in lags:
        Q = n * (n + 2) * np.sum(rho[:m] ** 2 / (n - np.arange(1, m + 1)))
        out[m] = {"Q": Q, "p-value": stats.chi2.sf(Q, m)}
    return pd.DataFrame(out).T

max_lag = max(LB_LAGS)

---
# THÀNH VIÊN 2: MÔ HÌNH HOÁ BIẾN ĐỘNG BIÊN (UNIVARIATE GARCH MARGINS)
## 2.1 Ước lượng GARCH(1,1) & GJR-GARCH(1,1) với phân phối Normal và Student-t
**Tầng 1 – Công thức & ý nghĩa:**
- Phương trình trung bình: $r_t = \mu + \epsilon_t$, $\epsilon_t = \sigma_t z_t$.
- GARCH(1,1): $\sigma_t^2 = \omega + \alpha\epsilon_{t-1}^2 + \beta\sigma_{t-1}^2$.
- GJR-GARCH(1,1): $\sigma_t^2 = \omega + \alpha\epsilon_{t-1}^2 + \gamma I_{t-1}\epsilon_{t-1}^2 + \beta\sigma_{t-1}^2$, $I_{t-1} = 1$ nếu $\epsilon_{t-1} < 0$.
- $\alpha$: phản ứng với tin tức mới; $\beta$: "trí nhớ" của biến động; persistence $= \alpha + \gamma/2 + \beta$ (GARCH: $\alpha+\beta$);
  half-life $= \ln 0{,}5/\ln(\text{persistence})$ phiên; phương sai dài hạn $\bar\sigma^2 = \omega/(1-\text{persistence})$.
- $\gamma > 0$: **hiệu ứng đòn bẩy** – tin xấu làm biến động tăng nhiều hơn tin tốt cùng độ lớn.
- $z_t \sim N(0,1)$ hoặc Student-t chuẩn hoá (phương sai 1, bậc tự do ν > 2) – t bắt đuôi dày mà TV1 đã chứng minh.
- Log-likelihood mỗi quan sát: Normal $\ell_t = -\tfrac12[\ln 2\pi + \ln\sigma_t^2 + \epsilon_t^2/\sigma_t^2]$;
  Student-t $\ell_t = \ln\Gamma(\tfrac{\nu+1}{2}) - \ln\Gamma(\tfrac{\nu}{2}) - \tfrac12\ln[\pi(\nu-2)] - \tfrac12\ln\sigma_t^2 - \tfrac{\nu+1}{2}\ln\left(1+\tfrac{\epsilon_t^2}{\sigma_t^2(\nu-2)}\right)$.

**Tầng 2:** `garch_variance()` lọc $\sigma_t^2$ (dùng `scipy.signal.lfilter`), `fit_garch()` tối đa hoá log-likelihood bằng
`scipy.optimize.minimize` (SLSQP, ràng buộc ω > 0, α ≥ 0, α + γ ≥ 0, β ≥ 0, persistence < 1, ν > 2). Sai số chuẩn robust
(Bollerslev–Wooldridge, sandwich). Ước lượng trên lợi suất % (r × 100); σ được đổi lại về thập phân khi lưu.

**Tầng 3:** p-value của γ < 0,05 ⇒ có đòn bẩy; ν nhỏ (3–6) ⇒ phần dư vẫn đuôi dày; persistence gần 1 ⇒ cú sốc biến động tắt rất chậm.

In [ ]:
# ===================== 2.1 HÀM GARCH / GJR-GARCH VIẾT TAY =====================
def unpack(params, gjr, dist):
    """Tách vector tham số [mu, omega, alpha, (gamma), beta, (nu)]."""
    mu, omega, alpha = params[0], params[1], params[2]
    gamma = params[3] if gjr else 0.0
    beta = params[4] if gjr else params[3]
    nu = params[-1] if dist == "t" else np.inf
    return mu, omega, alpha, gamma, beta, nu


def garch_variance(params, r, gjr, dist):
    """σ²_t = ω + (α + γ·I_{t-1}) ε²_{t-1} + β σ²_{t-1}; khởi tạo σ²_0 = phương sai mẫu (backcast)."""
    mu, omega, alpha, gamma, beta, _ = unpack(params, gjr, dist)
    eps = r - mu
    backcast = np.mean(eps ** 2)
    x = np.empty_like(eps)
    x[0] = omega + (alpha + 0.5 * gamma + beta) * backcast
    x[1:] = omega + (alpha + gamma * (eps[:-1] < 0)) * eps[:-1] ** 2
    sigma2 = lfilter([1.0], [1.0, -beta], x)               # σ²_t = x_t + β σ²_{t-1}
    return np.maximum(sigma2, 1e-12), eps


def loglik_t(params, r, gjr, dist):
    """Log-likelihood của từng quan sát (vector độ dài T)."""
    sigma2, eps = garch_variance(params, r, gjr, dist)
    if dist == "normal":
        return -0.5 * (np.log(2 * np.pi) + np.log(sigma2) + eps ** 2 / sigma2)
    nu = params[-1]
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2))
            - 0.5 * np.log(sigma2) - (nu + 1) / 2 * np.log1p(eps ** 2 / (sigma2 * (nu - 2))))


def robust_se(ll_fun, x, h=1e-4):
    """Sai số chuẩn sandwich A⁻¹BA⁻¹ (A: Hessian số, B: tích ngoài của score) – Bollerslev & Wooldridge (1992)."""
    x = np.asarray(x, float)
    k = len(x)
    step = h * np.maximum(np.abs(x), 1e-2)
    scores = np.empty((len(ll_fun(x)), k))
    for i in range(k):
        e = np.zeros(k); e[i] = step[i]
        scores[:, i] = (ll_fun(x + e) - ll_fun(x - e)) / (2 * step[i])
    L = lambda p: ll_fun(p).sum()
    Hs = np.empty((k, k))
    for i in range(k):
        for j in range(i, k):
            ei = np.zeros(k); ei[i] = step[i]
            ej = np.zeros(k); ej[j] = step[j]
            Hs[i, j] = Hs[j, i] = (L(x + ei + ej) - L(x + ei - ej) - L(x - ei + ej) + L(x - ei - ej)) / (4 * step[i] * step[j])
    try:
        A_inv = np.linalg.inv(-Hs)
        return np.sqrt(np.clip(np.diag(A_inv @ (scores.T @ scores) @ A_inv), 0, None))
    except np.linalg.LinAlgError:
        return np.full(k, np.nan)


def fit_garch(r, gjr=False, dist="normal"):
    """Ước lượng MLE; thử nhiều điểm xuất phát, giữ nghiệm có log-likelihood lớn nhất."""
    v = r.var()
    names = ["mu", "omega", "alpha"] + (["gamma"] if gjr else []) + ["beta"] + (["nu"] if dist == "t" else [])
    bounds = [(-5 * abs(r.mean()) - 1, 5 * abs(r.mean()) + 1), (1e-6, 10 * v), (0.0, 1.0)] \
        + ([(-0.5, 1.0)] if gjr else []) + [(0.0, 1.0)] + ([(2.05, 100.0)] if dist == "t" else [])
    ia, ib = 2, (4 if gjr else 3)
    cons = [{"type": "ineq", "fun": lambda p: 0.9999 - (p[ia] + (0.5 * p[3] if gjr else 0) + p[ib])}]
    if gjr:
        cons.append({"type": "ineq", "fun": lambda p: p[2] + p[3]})          # α + γ ≥ 0
    nll = lambda p: -loglik_t(p, r, gjr, dist).sum()
    best = None
    for a0, b0 in [(0.05, 0.90), (0.10, 0.85), (0.15, 0.75)]:
        x0 = [r.mean(), v * (1 - a0 - b0), a0] + ([0.05] if gjr else []) + [b0] + ([6.0] if dist == "t" else [])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = minimize(nll, x0, method="SLSQP", bounds=bounds, constraints=cons,
                           options={"maxiter": 2000, "ftol": 1e-10})
        if best is None or res.fun < best.fun:
            best = res
    x = best.x
    se = robust_se(lambda p: loglik_t(p, r, gjr, dist), x)
    se = np.where(se > 0, se, np.nan)
    p_val = 2 * stats.norm.sf(np.abs(x / se))
    sigma2, eps = garch_variance(x, r, gjr, dist)
    mu, omega, alpha, gamma, beta, nu = unpack(x, gjr, dist)
    pers = alpha + 0.5 * gamma + beta
    igarch = pers > 0.999                                         # chạm biên => IGARCH, σ dài hạn không tồn tại
    LL, k, n = -best.fun, len(x), len(r)
    sig2_next = omega + (alpha + gamma * (eps[-1] < 0)) * eps[-1] ** 2 + beta * sigma2[-1]
    return {"gjr": gjr, "dist": dist, "params": dict(zip(names, x)), "se": dict(zip(names, se)),
            "pvalue": dict(zip(names, p_val)), "LL": LL, "k": k, "AIC": -2 * LL + 2 * k, "BIC": -2 * LL + k * np.log(n),
            "persistence": pers, "IGARCH": igarch,
            "half_life": np.nan if igarch else np.log(0.5) / np.log(pers),
            "uncond_vol_ann": np.nan if igarch else np.sqrt(omega / (1 - pers) * TRADING_DAYS) / SCALE,
            "sigma": np.sqrt(sigma2) / SCALE, "std_resid": eps / np.sqrt(sigma2),
            "sigma_fcst": np.sqrt(sig2_next) / SCALE, "converged": bool(best.success)}

In [ ]:
# ===================== 2.1b ƯỚC LƯỢNG 4 MÔ HÌNH CHO MỖI MÃ =====================
SPECS = {"GARCH-N": dict(gjr=False, dist="normal"), "GARCH-t": dict(gjr=False, dist="t"),
         "GJR-GARCH-N": dict(gjr=True, dist="normal"), "GJR-GARCH-t": dict(gjr=True, dist="t")}
fits, param_rows = {}, []
for tk in tickers:
    r = log_ret[tk].values * SCALE
    for name, spec in SPECS.items():
        f = fit_garch(r, **spec)
        fits[(tk, name)] = f
        for pn, v in f["params"].items():
            param_rows.append({"Mã": tk, "Mô hình": name, "Tham số": pn, "Ước lượng": v,
                               "Std.Err (robust)": f["se"][pn], "p-value": f["pvalue"][pn]})
    print(f"✔ {tk}: đã ước lượng {len(SPECS)} mô hình")
all_params = pd.DataFrame(param_rows)
wide = all_params.pivot_table(index=["Mã", "Mô hình"], columns="Tham số", values="Ước lượng", sort=False)
display(wide[[c for c in ["mu", "omega", "alpha", "gamma", "beta", "nu"] if c in wide]].round(4))

## 2.2 Lựa chọn mô hình tối ưu (AIC, BIC, Log-Likelihood)
**Tầng 1:** $AIC = -2\ln L + 2k$, $BIC = -2\ln L + k\ln n$ – càng nhỏ càng tốt; BIC phạt số tham số mạnh hơn nên ưa mô hình gọn.
Kiểm định LR GARCH vs GJR (cùng phân phối): $LR = 2(LL_{GJR} - LL_{GARCH}) \sim \chi^2(1)$.

**Tầng 3:** Student-t có AIC/BIC thấp hơn hẳn Normal ⇒ phần dư vẫn đuôi dày sau khi đã mô hình hoá biến động.
GJR tốt hơn và LR có p < 0,05 ⇒ có hiệu ứng đòn bẩy. Mô hình có persistence ≥ 0,999 được đánh dấu **IGARCH** (cú sốc không tắt,
σ dài hạn không xác định) – vẫn dùng được để lọc σ_t nhưng không diễn giải σ dài hạn.

In [ ]:
# ===================== 2.2 SO SÁNH & CHỌN MÔ HÌNH =====================
comparison = pd.DataFrame([{
    "Mã": tk, "Mô hình": name, "LogLik": f["LL"], "k": f["k"], "AIC": f["AIC"], "BIC": f["BIC"],
    "Persistence": f["persistence"], "IGARCH?": "CÓ" if f["IGARCH"] else "",
    "Half-life (phiên)": f["half_life"], "σ dài hạn năm": f["uncond_vol_ann"],
    "gamma": f["params"].get("gamma", np.nan), "p(gamma)": f["pvalue"].get("gamma", np.nan),
    "nu": f["params"].get("nu", np.nan), "Hội tụ": f["converged"]} for (tk, name), f in fits.items()])
comparison["Hạng AIC"] = comparison.groupby("Mã")["AIC"].rank().astype(int)
comparison["Hạng BIC"] = comparison.groupby("Mã")["BIC"].rank().astype(int)
display(comparison.set_index(["Mã", "Mô hình"]).round(4))

lr_table = pd.DataFrame([{
    "Mã": tk, "Phân phối": d,
    "LR (GJR vs GARCH)": max(2 * (fits[(tk, f"GJR-GARCH-{d}")]["LL"] - fits[(tk, f"GARCH-{d}")]["LL"]), 0)}
    for tk in tickers for d in ["N", "t"]])
lr_table["p-value"] = stats.chi2.sf(lr_table["LR (GJR vs GARCH)"], 1)
lr_table["Kết luận"] = np.where(lr_table["p-value"] < ALPHA, "Có đòn bẩy (GJR tốt hơn)", "Không khác biệt")
display(lr_table.round(4))

best_name = comparison.sort_values(SELECT_BY).groupby("Mã", sort=False).head(1).set_index("Mã")["Mô hình"].loc[tickers]
selected = {tk: fits[(tk, best_name[tk])] for tk in tickers}
print(f"\nMô hình được chọn theo {SELECT_BY}:")
for tk in tickers:
    f = selected[tk]
    extra = ("IGARCH – σ dài hạn không xác định" if f["IGARCH"]
             else f"half-life = {f['half_life']:.1f} phiên | σ dài hạn năm = {f['uncond_vol_ann']:.2%}")
    print(f"  {tk}: {best_name[tk]:<12} | persistence = {f['persistence']:.4f} | {extra}")

## 2.3 Trích xuất phần dư chuẩn hoá ε và độ lệch chuẩn có điều kiện σ
**Tầng 1:** $\epsilon_{i,t} = \dfrac{r_{i,t} - \mu_i}{\sigma_{i,t}}$. Nếu mô hình đúng: ε có trung bình ≈ 0, phương sai ≈ 1, không còn cụm biến động.
$\sigma_{i,t}$ chỉ dùng thông tin đến $t-1$ ⇒ chính là **dự báo biến động 1 ngày** cho ngày t (dùng cho VaR ở TV4).

**Tầng 2:** Output dùng chung: `std_resid` (T × N), `cond_vol` (T × N, thập phân/ngày), `mu_vec` (N), `sigma_fcst` (σ phiên T+1).

**Tầng 3:** Biểu đồ: σ_t (đỏ) bám theo |r_t| (xám) và vọt lên trong các giai đoạn khủng hoảng; so sánh với EWMA (RiskMetrics λ = 0,94).

In [ ]:
# ===================== 2.3 LƯU ε, σ + BIỂU ĐỒ =====================
std_resid = pd.DataFrame({tk: selected[tk]["std_resid"] for tk in tickers}, index=log_ret.index)
cond_vol = pd.DataFrame({tk: selected[tk]["sigma"] for tk in tickers}, index=log_ret.index)
mu_vec = pd.Series({tk: selected[tk]["params"]["mu"] / SCALE for tk in tickers})      # μ thập phân
sigma_fcst = pd.Series({tk: selected[tk]["sigma_fcst"] for tk in tickers}, name="sigma_T+1")
print("Phần dư chuẩn hoá (kỳ vọng mean ≈ 0, std ≈ 1):")
resid_stats = std_resid.agg(["mean", "std", "skew", "kurt"]).T
display(resid_stats.round(4))
low_std = resid_stats.index[resid_stats["std"] < 0.95].tolist()
if low_std:
    print("Lưu ý: std(ε) < 0,95 ở " + ", ".join(low_std) + " -> σ_t hơi cao hơn biến động thực hiện "
          "(thường gặp khi persistence sát 1 / IGARCH hoặc phân phối t với ν nhỏ) -> VaR từ các mã này có xu hướng thận trọng.")
print("σ dự báo phiên T+1 (%/ngày):", (sigma_fcst * 100).round(3).to_dict())

ewma = log_ret.pow(2).ewm(alpha=1 - 0.94, adjust=False).mean().shift(1).pipe(np.sqrt)
fig, axes = grid_axes(N, sharex=True)
for ax, tk in zip(axes, tickers):
    ax.plot(log_ret.index, log_ret[tk].abs() * np.sqrt(TRADING_DAYS), color="lightgrey", lw=0.5, label="|r_t| năm hoá")
    ax.plot(cond_vol.index, cond_vol[tk] * np.sqrt(TRADING_DAYS), color="tab:red", lw=1, label="σ_t mô hình được chọn")
    ax.plot(ewma.index, ewma[tk] * np.sqrt(TRADING_DAYS), color="tab:blue", lw=0.8, ls="--", label="EWMA λ=0,94")
    shade_stress(ax, alpha=0.08, label=False)
    ax.set_title(f"{tk} – σ_t năm hoá ({best_name[tk]})"); ax.set_ylim(0, None); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.savefig(OUT["TV2"] / "2_conditional_volatility.png", dpi=150, bbox_inches="tight"); plt.show()

# News Impact Curve: σ²_t theo cú sốc ε_{t-1}, giữ σ²_{t-1} = phương sai mẫu.
# Dùng GJR-GARCH-t cho CẢ 5 mã để so sánh mức bất đối xứng; mã có mô hình được chọn KHÔNG phải GJR (γ không có ý nghĩa)
# vẽ nét đứt – với các mã này đường cong gần đối xứng, đúng với kết luận "không có hiệu ứng đòn bẩy".
fig, ax = plt.subplots(figsize=(9, 4.8))
shocks = np.linspace(-5, 5, 201)
for tk in tickers:
    f = fits[(tk, "GJR-GARCH-t")]
    p = f["params"]
    nic = p["omega"] + (p["alpha"] + p["gamma"] * (shocks < 0)) * shocks ** 2 + p["beta"] * (log_ret[tk] * SCALE).var()
    chosen_gjr = best_name[tk].startswith("GJR")
    ax.plot(shocks, nic - nic.min(), ls="-" if chosen_gjr else "--",
            label=f"{tk}: γ = {p['gamma']:.3f} (p = {f['pvalue']['gamma']:.2f})"
                  + ("" if chosen_gjr else f" – chọn {best_name[tk]}"))
ax.axvline(0, color="k", lw=0.5); ax.grid(alpha=0.3); ax.legend(fontsize=8)
ax.set_title("News Impact Curve – GJR-GARCH-t (đã trừ mức nền; nét đứt = mã không chọn GJR)")
ax.set_xlabel("Cú sốc ε_{t-1} (% lợi suất)"); ax.set_ylabel("Δσ²_t (%²)")
plt.tight_layout(); plt.savefig(OUT["TV2"] / "2_news_impact_curve.png", dpi=150); plt.show()

## 2.4 Chẩn đoán phần dư (Residual Diagnostics)
**Tầng 1:** Ljung-Box trên ε kiểm tra tự tương quan còn sót trong phương trình trung bình; Ljung-Box trên ε² kiểm tra **hiệu ứng ARCH còn sót**.
So với $r_t^2$ ở TV1 (p ≈ 0): nếu p của ε² > 0,05 ⇒ GARCH đã "hấp thụ" cụm biến động.

**Tầng 3:** Yêu cầu của đề: p ≥ 0,05 ở cả ε và ε². Nếu một mã còn p < 0,05 ở ε (mức sát ngưỡng) nhưng ε² đạt, phần biến động đã được
mô hình hoá tốt; phần tự tương quan nhỏ còn lại nằm ở phương trình trung bình (có thể thêm AR(1) nếu muốn loại bỏ hoàn toàn).
So sánh với TV1: tự tương quan của r_t ở VCB/VHM có thể biến mất sau khi chuẩn hoá bằng σ_t (do các ngày biến động lớn chi phối Q-stat).

In [ ]:
# ===================== 2.4 LJUNG-BOX TRÊN ε VÀ ε² =====================
diag_rows = []
for tk in tickers:
    for label, s in [("r_t² (trước GARCH)", log_ret[tk] ** 2), ("ε", std_resid[tk]), ("ε²", std_resid[tk] ** 2)]:
        for m, row in ljung_box(s, LB_LAGS).iterrows():
            diag_rows.append({"Mã": tk, "Chuỗi": label, "Lag": int(m), "Q": row["Q"], "p-value": row["p-value"]})
diagnostics = pd.DataFrame(diag_rows)
diag_pivot = diagnostics.pivot_table(index=["Mã", "Chuỗi"], columns="Lag", values="p-value", sort=False)
diag_pivot.columns = [f"p (lag {c})" for c in diag_pivot.columns]
display(diag_pivot.round(4))
at = lambda lab: diagnostics[(diagnostics["Chuỗi"] == lab) & (diagnostics["Lag"] == max_lag)].set_index("Mã")["p-value"]
print(f"ε  (lag {max_lag}): " + ", ".join(f"{tk}={'còn tự tương quan' if p < ALPHA else 'đạt'} (p={p:.3f})" for tk, p in at("ε").items()))
print(f"ε² (lag {max_lag}): " + ", ".join(f"{tk}={'CÒN ARCH' if p < ALPHA else 'đã loại bỏ ARCH'} (p={p:.3f})" for tk, p in at("ε²").items()))

# ---- Lưu kết quả TV2 ----
# Lưu ý đơn vị: mô hình ước lượng trên lợi suất % (r × 100) -> mu tính bằng %/ngày, omega bằng %²
UNIT = {"mu": "mu (%/ngày)", "omega": "omega (%²)"}
selected_df = pd.DataFrame({tk: {
    "Mô hình": best_name[tk], **{UNIT.get(k, k): v for k, v in selected[tk]["params"].items()},
    "LogLik": selected[tk]["LL"], "k": selected[tk]["k"], "AIC": selected[tk]["AIC"], "BIC": selected[tk]["BIC"],
    "Persistence": selected[tk]["persistence"], "IGARCH?": "CÓ" if selected[tk]["IGARCH"] else "không",
    "Half-life (phiên)": selected[tk]["half_life"],
    "σ dài hạn năm (%)": selected[tk]["uncond_vol_ann"] * 100,
    "σ T+1 (%/ngày)": sigma_fcst[tk] * 100,
    "std(ε)": std_resid[tk].std()} for tk in tickers}).T
display(selected_df)
std_resid.to_csv(OUT["TV2"] / "std_resid.csv", encoding="utf-8-sig")
cond_vol.to_csv(OUT["TV2"] / "cond_vol.csv", encoding="utf-8-sig")
with pd.ExcelWriter(OUT["TV2"] / "TV2_garch_results.xlsx", engine="openpyxl") as xw:
    selected_df.to_excel(xw, sheet_name="Mo_hinh_chon")
    comparison.to_excel(xw, sheet_name="So_sanh_AIC_BIC", index=False)
    lr_table.to_excel(xw, sheet_name="LR_GJR_vs_GARCH", index=False)
    all_params.to_excel(xw, sheet_name="Tham_so_day_du", index=False)
    diagnostics.to_excel(xw, sheet_name="Ljung_Box_phan_du", index=False)
print("✔ Đã lưu kết quả TV2 vào", OUT["TV2"])
# ---- File bàn giao cho TV3, TV4, TV5 (mô hình biên đã chọn) ----
handoff = pd.DataFrame({tk: {"Mô hình": best_name[tk], "gjr": SPECS[best_name[tk]]["gjr"], "dist": SPECS[best_name[tk]]["dist"],
                             "mu": mu_vec[tk], "k": selected[tk]["k"], "converged": bool(selected[tk]["converged"]),
                             "sigma_T+1": sigma_fcst[tk]} for tk in tickers}).T
handoff.index.name = "Mã"
handoff.to_csv(OUT["TV2"] / "garch_selected_models.csv", encoding="utf-8-sig")
print("✔ Đã lưu file bàn giao", OUT["TV2"] / "garch_selected_models.csv")


## 2.5 Diễn giải cốt lõi – vai trò của GARCH đơn biến trong DCC hai bước
DCC-GARCH (Engle, 2002) tách log-likelihood thành **phần biến động** + **phần tương quan**:
1. **Bước 1 (TV2):** GARCH riêng cho từng mã ⇒ có $\sigma_{i,t}$, **chuẩn hoá biến động riêng lẻ**, loại bỏ phương sai thay đổi theo thời gian.
2. **Bước 2 (TV3):** dùng $\epsilon_{i,t}$ (phương sai ≈ 1, không còn ARCH) làm đầu vào để ước lượng tương quan động $R_t$.

Nếu bỏ bước 1, hai mã cùng tăng biến động sẽ trông như tương quan tăng ⇒ tương quan ước lượng bị nhiễu bởi biến động.

### TV2 – Gợi ý trả lời vấn đáp
**Câu 1 – α + β ≈ 1 nghĩa là gì?** Biến động có **độ bền rất cao**: cú sốc hôm nay ảnh hưởng tới biến động nhiều tuần sau (half-life dài),
phương sai hội tụ rất chậm về mức dài hạn. Khi α + β = 1 là IGARCH (giống EWMA) – cú sốc không bao giờ tắt và phương sai dài hạn không
tồn tại (trường hợp được đánh dấu IGARCH ở bảng trên). Hệ quả: dự báo VaR nhiều ngày phải dùng σ hiện tại, không thể giả định nhanh chóng quay về mức bình thường.

**Câu 2 – Hiệu ứng đòn bẩy ở Việt Nam?** γ > 0 ⇒ phiên giảm điểm làm biến động phiên sau tăng mạnh hơn phiên tăng cùng biên độ.
Thị trường Việt Nam có tỷ trọng nhà đầu tư cá nhân lớn và dùng margin nhiều: giá giảm ⇒ call margin ⇒ bán giải chấp dây chuyền ⇒ biến động
tăng (rõ nhất năm 2022). Về quản trị: VaR cần phản ứng nhanh hơn sau phiên giảm mạnh.